In [6]:
import os
import sys
import time

from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassificationModel, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Start overall model pipeline timer
model_pipeline_start = time.time()

# 1. Initialize Spark Session fresh
spark = SparkSession.builder \
    .appName("AirQualityMLTraining") \
    .master("local[*]") \
    .getOrCreate()

print("🏁 Spark Session successfully initialized over a clean network port!")

try:
    # 2. Load the fallback CSV files natively
    spark_train = spark.read.csv("clean_air_quality_for_plots.csv", header=True, inferSchema=True)
    spark_test = spark.read.csv("clean_air_quality_for_ml_testing.csv", header=True, inferSchema=True)
    
    # 3. Compile structural feature matrices dynamically
    from pyspark.ml.feature import VectorAssembler
    features_to_vectorize = [
        'components_co', 'components_no2', 'components_o3', 'components_pm2_5', 
        'components_pm10', 'components_so2', 'temperature_2m', 
        'relative_humidity_2m', 'wind_speed_10m', 'hour', 'month'
    ]
    input_features = [col for col in features_to_vectorize if col in spark_train.columns]
    assembler = VectorAssembler(inputCols=input_features, outputCol="features")
    
    train_data = assembler.transform(spark_train).select("features", "main_aqi")
    test_data = assembler.transform(spark_test).select("features", "main_aqi")
    print("📊 Clean vector data matrices compiled successfully.")

    # 4. Train the Random Forest Classifier
    print("\n🤖 Training the Random Forest Classifier (Trees=15, Depth=5)...")
    train_start = time.time()
    
    rf = RandomForestClassifier(
        labelCol="main_aqi", 
        featuresCol="features", 
        numTrees=15, 
        maxDepth=5, 
        seed=42
    )
    
    rf_model = rf.fit(train_data)
    training_duration = time.time() - train_start
    print(f"✅ Model training completed in {training_duration:.2f} seconds.")

    # 5. Run Evaluation Predictions
    print("🔮 Running predictive analysis on test split...")
    predictions = rf_model.transform(test_data)

    # 6. Calculate Performance Metrics
    evaluator_acc = MulticlassClassificationEvaluator(labelCol="main_aqi", predictionCol="prediction", metricName="accuracy")
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="main_aqi", predictionCol="prediction", metricName="f1")
    
    accuracy = evaluator_acc.evaluate(predictions)
    f1_score = evaluator_f1.evaluate(predictions)
    
    print("\n==================================================")
    print("🏆 SPARK ML MULTICLASS CLASSIFICATION RESULTS")
    print("==================================================")
    print(f"✨ MODEL ACCURACY:           {accuracy * 100:.2f}%")
    print(f"✨ F1-SCORE VALUE:          {f1_score * 100:.2f}%")
    print("==================================================")

    # 7. EXPLICIT NATIVE SAVE (Bypassing pickle and pipeline structures entirely)
    print("\n💾 Writing model binary payload directly to storage...")
    
    # We call save directly on the trained model object using a clean folder target
    model_save_path = "rf_air_quality_model_spark"
    rf_model.save(model_save_path)
    
    print(f"🎉 SUCCESS: Model state directory saved to '{model_save_path}'!")

except Exception as e:
    print(f"\n❌ Error during execution pipeline: {e}")

finally:
    spark.stop()
    print("🛑 Spark active worker session closed safely.")

🏁 Spark Session successfully initialized over a clean network port!
📊 Clean vector data matrices compiled successfully.

🤖 Training the Random Forest Classifier (Trees=15, Depth=5)...
✅ Model training completed in 4.15 seconds.
🔮 Running predictive analysis on test split...

🏆 SPARK ML MULTICLASS CLASSIFICATION RESULTS
✨ MODEL ACCURACY:           71.49%
✨ F1-SCORE VALUE:          69.08%

💾 Writing model binary payload directly to storage...
🎉 SUCCESS: Model state directory saved to 'rf_air_quality_model_spark'!
🛑 Spark active worker session closed safely.
